# Phase 09 — Grounded prompt engineering

This notebook designs and compares eight prompt-engineering techniques for the Business Knowledge AI RAG workflow. It uses only real BM25-retrieved OpenStax chunks produced in Phase 07 as its proposed answer context. The intended generator is exactly Qwen2.5-7B-Instruct; no alternate LLM is selected.

> Scope boundary: This is prompt construction and an optional single-pass generation experiment only. It contains no LangGraph, autonomous agent, tool call, retrieval loop, or self-correction loop.

The OpenStax attribution notice recorded in the project documentation requires explicit OpenStax permission before the textbook is ingested into a generative-AI offering. Therefore, the notebook refuses to send textbook context to an LLM unless the operator has explicitly set OPENSTAX_GENERATIVE_AI_PERMISSION_CONFIRMED=true and the exact required model is available in the live sandbox catalog.


In [1]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Business Knowledge AI project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BM25_RESULTS_PATH = PROCESSED_DIR / "introduction_to_business_bm25_retrieval_results.json"
PROMPT_RESULTS_PATH = PROCESSED_DIR / "introduction_to_business_prompt_engineering_results.json"
STATUS_PATH = PROCESSED_DIR / "introduction_to_business_prompt_engineering_status.json"

REQUIRED_MODEL_ID = "Qwen2.5-7B-Instruct"
QUESTION = "What role do small businesses play in the U.S. economy?"
print(f"Project root: {PROJECT_ROOT}")
print(f"Required generator: {REQUIRED_MODEL_ID}")


Project root: /home/ubuntu/business-knowledge-ai
Required generator: Qwen2.5-7B-Instruct


## 1. Load real retrieved business context

Dense and fused-hybrid retrieval remain unavailable because Phase 04 did not produce real BGE-M3 vectors. Phase 07 did, however, execute BM25 over 3,018 real chunk records. This notebook uses the existing BM25 Top-3 response for the closely matched economy question as context; it does not invent a hybrid candidate set or reranked context.


In [2]:
with BM25_RESULTS_PATH.open("r", encoding="utf-8") as handle:
    bm25_runs = json.load(handle)

context_run = next(
    run
    for run in bm25_runs
    if run["query_id"] == "business-foundations" and run["top_k"] == 3
)

context_items = context_run["results"]
assert len(context_items) == 3, "Expected the real BM25 Top-3 context set."

context_blocks = []
for item in context_items:
    chapter = item["chapter"] or {}
    section = item["section"] or {}
    label = (
        f"[{item['chunk_id']} | page {item['page']} | "
        f"chapter {chapter.get('number', 'unknown')} | "
        f"section {section.get('number', 'not detected')} | "
        f"BM25 {item['bm25_score']:.4f}]"
    )
    context_blocks.append(f"{label}\n{item['text']}")

RETRIEVED_CONTEXT = "\n\n---\n\n".join(context_blocks)

context_preview = pd.DataFrame(
    [
        {
            "rank": item["rank"],
            "chunk_id": item["chunk_id"],
            "page": item["page"],
            "chapter": item["chapter"]["title"] if item["chapter"] else None,
            "section": item["section"]["number"] if item["section"] else None,
            "bm25_score": round(item["bm25_score"], 4),
            "text_preview": item["text"][:220].replace("\n", " ") + "…",
        }
        for item in context_items
    ]
)
display(context_preview)
display(Markdown("### Retrieved context supplied to every prompt"))
print(RETRIEVED_CONTEXT)


,rank,chunk_id,page,chapter,section,bm25_score,text_preview
0,1,openstax-introduction-business-p0193-c001,193,Entrepreneurship: Starting and Managing Your O...,5.4,20.1628,Chapter 5 Entrepreneurship: Starting and Manag...
1,2,openstax-introduction-business-p0157-c002,157,Forms of Business Ownership,NaN,18.2277,Lacks continuity when owner dies. Table 4.4 C ...
2,3,openstax-introduction-business-p0216-c001,216,Entrepreneurship: Starting and Managing Your O...,5.8,17.2205,204 Chapter 5 Entrepreneurship: Starting and M...


### Retrieved context supplied to every prompt

[openstax-introduction-business-p0193-c001 | page 193 | chapter 5 | section 5.4 | BM25 20.1628]
Chapter 5 Entrepreneurship: Starting and Managing Your Own Business
181
C O N C E P T C H E C K
1.
Describe the personality traits and skills characteristic of successful entrepreneurs.
2.
What does it mean when we say that an entrepreneur should work on the business, not in it?
5.3
Small Business: Driving America's Growth
3.
How do small businesses contribute to the U.S. economy?
Although large corporations dominated the business scene for many decades, in recent years small
businesses have once again come to the forefront. Downsizings that accompany economic downturns have
caused many people to look toward smaller companies for employment, and they have plenty to choose from.
Small businesses play an important role in the U.S. economy, representing about half of U.S. economic output,

---

[openstax-introduction-business-p0157-c002 | page 157 | chapter 4 | section not detected | BM25 18.22

## 2. Exact-model and permission preflight

The preflight asks the live sandbox catalog whether the exact requested model identifier exists. A close or differently named model is not accepted. It separately requires a positive, explicit permission confirmation; an absent confirmation is treated as no permission. This avoids silently substituting a model or transmitting textbook context to a generative model without the required authorization.


In [3]:
def live_model_ids() -> tuple[list[str], str | None]:
    api_base = os.environ.get("OPENAI_API_BASE")
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_base or not api_key:
        return [], "OPENAI_API_BASE or OPENAI_API_KEY is unavailable."

    request = Request(
        f"{api_base.rstrip('/')}/models",
        headers={"Authorization": f"Bearer {api_key}"},
    )
    try:
        with urlopen(request, timeout=30) as response:
            payload = json.loads(response.read().decode("utf-8"))
        return [model["id"] for model in payload.get("data", []) if "id" in model], None
    except (URLError, TimeoutError, json.JSONDecodeError) as exc:
        return [], f"Catalog request failed: {type(exc).__name__}: {exc}"


catalog_ids, catalog_error = live_model_ids()
matching_qwen_ids = [model_id for model_id in catalog_ids if "qwen" in model_id.lower()]
exact_model_available = REQUIRED_MODEL_ID in catalog_ids
permission_confirmed = os.environ.get(
    "OPENSTAX_GENERATIVE_AI_PERMISSION_CONFIRMED", ""
).strip().lower() == "true"
execution_allowed = exact_model_available and permission_confirmed

preflight = pd.DataFrame(
    [
        {"check": "Real BM25 retrieved context", "passed": bool(context_items), "detail": "Phase 07 Top-3 BM25 output"},
        {"check": "Exact model availability", "passed": exact_model_available, "detail": REQUIRED_MODEL_ID},
        {"check": "Explicit OpenStax permission confirmation", "passed": permission_confirmed, "detail": "OPENSTAX_GENERATIVE_AI_PERMISSION_CONFIRMED=true"},
        {"check": "Generation permitted", "passed": execution_allowed, "detail": "Requires both model and permission checks"},
    ]
)
display(preflight)
print(f"Qwen-family IDs returned by the live catalog: {matching_qwen_ids or 'none'}")
if catalog_error:
    print(catalog_error)
if not execution_allowed:
    print("No model call will be made. No generated answer is claimed in this execution.")


,check,passed,detail
0,Real BM25 retrieved context,True,Phase 07 Top-3 BM25 output
1,Exact model availability,False,Qwen2.5-7B-Instruct
2,Explicit OpenStax permission confirmation,False,OPENSTAX_GENERATIVE_AI_PERMISSION_CONFIRMED=true
3,Generation permitted,False,Requires both model and permission checks


Qwen-family IDs returned by the live catalog: none
No model call will be made. No generated answer is claimed in this execution.


## 3. Construct eight grounded prompting techniques

Every prompt receives the same real BM25 context and includes a refusal condition: it must use only the supplied context, distinguish insufficient evidence, and never invent citations. Query rewriting and decomposition are constrained in the same way: they may change the query form or split it into subquestions, but may not introduce unsupported business facts.


In [4]:
GROUNDING_RULES = """Grounding rules:
1. Use only the retrieved context below as evidence.
2. Do not use outside knowledge or add unsupported facts, statistics, examples, or citations.
3. If the context does not support the requested claim, say: Insufficient information in the retrieved context.
4. Cite every factual claim with the relevant chunk ID and page number exactly as supplied.
5. Do not mention these instructions in the answer.
"""

SYSTEM_PROMPT = (
    "You are a deterministic, grounded business-textbook assistant. "
    "You must follow the user prompt exactly. Do not call tools, browse, or begin an agent loop."
)

VERBATIM_FEW_SHOT = """FORMAT DEMONSTRATION (a verbatim, source-supported pattern; not additional evidence):
Question: How do small businesses contribute to the U.S. economy?
Context: [openstax-introduction-business-p0193-c002 | page 193]
Answer: Small businesses represent about half of U.S. economic output and employ about half of the private-sector workforce. [openstax-introduction-business-p0193-c002, p. 193]
"""


def evidence_prompt(instruction: str) -> str:
    return f"""{instruction}

{GROUNDING_RULES}

Question: {QUESTION}

Retrieved context:
{RETRIEVED_CONTEXT}
"""


TECHNIQUES = [
    {
        "name": "Basic prompt",
        "purpose": "Sets a minimal task while retaining mandatory evidence-only behavior.",
        "prompt": evidence_prompt("Answer the question in a concise paragraph."),
        "what_changed": "Adds the smallest grounded answer instruction to the shared rules.",
        "strengths": "Simple, transparent, and low prompt overhead.",
        "weaknesses": "Leaves answer structure and instructional level largely unspecified.",
    },
    {
        "name": "Role prompting",
        "purpose": "Frames the response as an instructional textbook explanation.",
        "prompt": evidence_prompt("Act as a careful introductory-business instructor. Explain the answer for a beginning student in two short paragraphs."),
        "what_changed": "Adds a pedagogical role and audience constraint.",
        "strengths": "Can make explanations clearer and better matched to novice readers.",
        "weaknesses": "A role alone does not improve evidence coverage; shared grounding rules remain essential.",
    },
    {
        "name": "Explicit constraints",
        "purpose": "Makes output length, coverage, and abstention requirements testable.",
        "prompt": evidence_prompt("Provide exactly 3 bullet points. Each bullet must make one source-supported claim and include at least one chunk-and-page citation. Do not exceed 120 words."),
        "what_changed": "Adds explicit form, length, citation, and claim-count constraints.",
        "strengths": "Produces a more predictable, reviewable response format.",
        "weaknesses": "Tight limits can omit relevant nuance when the context is broad.",
    },
    {
        "name": "Grounded prompting",
        "purpose": "Requires an evidence-first answer plan and visible support mapping.",
        "prompt": evidence_prompt("First state the evidence-supported answer. Then add an Evidence used section that lists only the chunk IDs and pages that directly support the answer."),
        "what_changed": "Explicitly separates the answer from its supporting evidence references.",
        "strengths": "Makes provenance inspection easier and discourages unsupported expansion.",
        "weaknesses": "Adds response length and may repeat citation information.",
    },
    {
        "name": "Few-shot prompting",
        "purpose": "Demonstrates a grounded answer-and-citation format before the target question.",
        "prompt": evidence_prompt(f"{VERBATIM_FEW_SHOT}\nNow answer the target question using the same evidence-limited format."),
        "what_changed": "Adds a verbatim, source-supported formatting example before the target task.",
        "strengths": "Clarifies expected citation placement and response shape.",
        "weaknesses": "Consumes context budget and can anchor outputs too closely to the example format.",
    },
    {
        "name": "Citation prompting",
        "purpose": "Imposes claim-level citation traceability.",
        "prompt": evidence_prompt("Write one concise answer. After every factual sentence, append citations in this exact format: [chunk_id, p. page]. Do not cite a chunk unless it supports that sentence."),
        "what_changed": "Requires a defined citation syntax at the factual-sentence level.",
        "strengths": "Facilitates citation auditing and downstream inline-citation rendering.",
        "weaknesses": "Frequent citations may reduce readability and reveal retrieval gaps.",
    },
    {
        "name": "Query rewriting",
        "purpose": "Rephrases the user question for retrieval without introducing new facts.",
        "prompt": evidence_prompt("Rewrite the question into one concise retrieval query. Preserve its intent, add no factual terms that are not present in the question or retrieved context, and output only the rewritten query with no answer."),
        "what_changed": "Changes the expected output from an answer to a grounded retrieval-query rewrite.",
        "strengths": "Can standardize vocabulary for a later deterministic retrieval pass.",
        "weaknesses": "Must be evaluated before use because an unnecessary rewrite can narrow or distort retrieval intent.",
    },
    {
        "name": "Query decomposition",
        "purpose": "Breaks a broad question into evidence-bounded subquestions.",
        "prompt": evidence_prompt("Decompose the question into at most 3 independent subquestions that can be answered from the retrieved context. Do not answer them. Do not introduce entities, causes, or facts absent from the question or context."),
        "what_changed": "Changes the expected output into a short, evidence-bounded subquestion plan.",
        "strengths": "Can expose distinct facets for a later fixed retrieval-and-answer workflow.",
        "weaknesses": "Adds orchestration complexity and must remain a fixed, non-agentic step.",
    },
]

assert len(TECHNIQUES) == 8
assert all(RETRIEVED_CONTEXT in technique["prompt"] for technique in TECHNIQUES)
assert all("Insufficient information" in technique["prompt"] for technique in TECHNIQUES)
print(f"Prepared {len(TECHNIQUES)} grounded prompt variants.")


Prepared 8 grounded prompt variants.


## 4. Optional exact-model execution

The function below is the real, single-pass model invocation path. It has no fallback model and never invokes tools or LangGraph. In this environment the preflight determines whether it is legal and technically possible to run. When blocked, the displayed “generated answer” is an explicit non-generation record rather than a substitute response.


In [5]:
def generate_with_exact_model(prompt: str) -> dict:
    if not execution_allowed:
        reasons = []
        if not exact_model_available:
            reasons.append(f"the exact model {REQUIRED_MODEL_ID!r} is not available in the live catalog")
        if not permission_confirmed:
            reasons.append("OpenStax generative-AI permission has not been explicitly confirmed")
        return {
            "generation_status": "not_generated_preflight_blocked",
            "answer": "NOT GENERATED — " + "; ".join(reasons) + ".",
            "model": None,
            "usage": None,
        }

    from openai import OpenAI

    client = OpenAI()
    response = client.chat.completions.create(
        model=REQUIRED_MODEL_ID,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        max_tokens=600,
    )
    return {
        "generation_status": "generated_exact_model",
        "answer": response.choices[0].message.content,
        "model": REQUIRED_MODEL_ID,
        "usage": {
            "prompt_tokens": getattr(response.usage, "prompt_tokens", None),
            "completion_tokens": getattr(response.usage, "completion_tokens", None),
            "total_tokens": getattr(response.usage, "total_tokens", None),
        },
    }


In [6]:
experiment_records = []

for number, technique in enumerate(TECHNIQUES, start=1):
    generated = generate_with_exact_model(technique["prompt"])
    display(Markdown(f"## {number}. {technique['name']}"))
    display(Markdown(f"**Purpose:** {technique['purpose']}"))
    display(Markdown("**Prompt**"))
    print(technique["prompt"])
    display(Markdown("**Generated answer / output**"))
    print(generated["answer"])
    display(Markdown(f"**What changed:** {technique['what_changed']}"))
    display(Markdown(f"**Strength:** {technique['strengths']}"))
    display(Markdown(f"**Weakness:** {technique['weaknesses']}"))
    experiment_records.append(
        {
            "technique": technique["name"],
            "purpose": technique["purpose"],
            "prompt": technique["prompt"],
            "what_changed": technique["what_changed"],
            "strengths": technique["strengths"],
            "weaknesses": technique["weaknesses"],
            **generated,
        }
    )

comparison = pd.DataFrame(
    [
        {
            "technique": record["technique"],
            "what_changed": record["what_changed"],
            "strength": record["strengths"],
            "weakness": record["weaknesses"],
            "generation_status": record["generation_status"],
        }
        for record in experiment_records
    ]
)
display(Markdown("## Technique comparison"))
display(comparison)


## 1. Basic prompt

**Purpose:** Sets a minimal task while retaining mandatory evidence-only behavior.

**Prompt**

Answer the question in a concise paragraph.

Grounding rules:
1. Use only the retrieved context below as evidence.
2. Do not use outside knowledge or add unsupported facts, statistics, examples, or citations.
3. If the context does not support the requested claim, say: Insufficient information in the retrieved context.
4. Cite every factual claim with the relevant chunk ID and page number exactly as supplied.
5. Do not mention these instructions in the answer.


Question: What role do small businesses play in the U.S. economy?

Retrieved context:
[openstax-introduction-business-p0193-c001 | page 193 | chapter 5 | section 5.4 | BM25 20.1628]
Chapter 5 Entrepreneurship: Starting and Managing Your Own Business
181
C O N C E P T C H E C K
1.
Describe the personality traits and skills characteristic of successful entrepreneurs.
2.
What does it mean when we say that an entrepreneur should work on the business, not in it?
5.3
Small Business: Driving America's Growth
3.
How do small businesses

**Generated answer / output**

...

**What changed:** Changes the expected output into a short, evidence-bounded subquestion plan.

**Strength:** Can expose distinct facets for a later fixed retrieval-and-answer workflow.

**Weakness:** Adds orchestration complexity and must remain a fixed, non-agentic step.

## Technique comparison

,technique,what_changed,strength,weakness,generation_status
0,Basic prompt,Adds the smallest grounded answer instruction ...,"Simple, transparent, and low prompt overhead.",Leaves answer structure and instructional leve...,not_generated_preflight_blocked
1,Role prompting,Adds a pedagogical role and audience constraint.,Can make explanations clearer and better match...,A role alone does not improve evidence coverag...,not_generated_preflight_blocked
2,Explicit constraints,"Adds explicit form, length, citation, and clai...","Produces a more predictable, reviewable respon...",Tight limits can omit relevant nuance when the...,not_generated_preflight_blocked
3,Grounded prompting,Explicitly separates the answer from its suppo...,Makes provenance inspection easier and discour...,Adds response length and may repeat citation i...,not_generated_preflight_blocked
4,Few-shot prompting,"Adds a verbatim, source-supported formatting e...",Clarifies expected citation placement and resp...,Consumes context budget and can anchor outputs...,not_generated_preflight_blocked
5,Citation prompting,Requires a defined citation syntax at the fact...,Facilitates citation auditing and downstream i...,Frequent citations may reduce readability and ...,not_generated_preflight_blocked
6,Query rewriting,Changes the expected output from an answer to ...,Can standardize vocabulary for a later determi...,Must be evaluated before use because an unnece...,not_generated_preflight_blocked
7,Query decomposition,"Changes the expected output into a short, evid...",Can expose distinct facets for a later fixed r...,Adds orchestration complexity and must remain ...,not_generated_preflight_blocked


## 5. Persist the auditable experiment record

The artifact preserves the exact prompts, grounding rules, real BM25 context metadata, preflight outcome, and either genuine exact-model output or an explicit non-generation reason. The comparative notes describe prompt-design trade-offs only; they are not claims of observed model-quality improvements when generation has not run.


In [7]:
prompt_results = {
    "phase": "09_prompt_engineering",
    "question": QUESTION,
    "required_model": REQUIRED_MODEL_ID,
    "system_prompt": SYSTEM_PROMPT,
    "grounding_rules": GROUNDING_RULES,
    "retrieval_context": {
        "source_artifact": str(BM25_RESULTS_PATH.relative_to(PROJECT_ROOT)),
        "query_id": context_run["query_id"],
        "retrieval_question": context_run["question"],
        "top_k": context_run["top_k"],
        "chunks": [
            {
                "chunk_id": item["chunk_id"],
                "page": item["page"],
                "chapter": item["chapter"],
                "section": item["section"],
                "bm25_score": item["bm25_score"],
            }
            for item in context_items
        ],
    },
    "experiments": experiment_records,
}

with PROMPT_RESULTS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(prompt_results, handle, indent=2, ensure_ascii=False)

status = {
    "phase": "09_prompt_engineering",
    "required_model": REQUIRED_MODEL_ID,
    "model_catalog_checked": True,
    "exact_model_available": exact_model_available,
    "matching_qwen_model_ids": matching_qwen_ids,
    "openstax_generative_ai_permission_confirmed": permission_confirmed,
    "retrieved_context_source": str(BM25_RESULTS_PATH.relative_to(PROJECT_ROOT)),
    "retrieval_type": "BM25 only",
    "retrieved_context_is_real": True,
    "hybrid_or_reranked_context_available": False,
    "prompt_techniques": [technique["name"] for technique in TECHNIQUES],
    "prompt_count": len(TECHNIQUES),
    "model_invocation_implemented": True,
    "model_invocation_executed": any(
        record["generation_status"] == "generated_exact_model" for record in experiment_records
    ),
    "generated_answer_count": sum(
        record["generation_status"] == "generated_exact_model" for record in experiment_records
    ),
    "results_artifact": str(PROMPT_RESULTS_PATH.relative_to(PROJECT_ROOT)),
    "langgraph_implemented": False,
    "agentic_control_flow_implemented": False,
    "no_model_substitution": True,
    "no_fabricated_answers": True,
    "status": (
        "completed_exact_model_generation"
        if execution_allowed
        else "blocked_exact_model_or_permission_preflight"
    ),
    "limitation": (
        None
        if execution_allowed
        else "No Qwen2.5-7B-Instruct generation was run. Execution requires both the exact model in the live catalog and explicit OpenStax permission confirmation; no substitute model or fabricated answer was used."
    ),
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
}

with STATUS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(status, handle, indent=2, ensure_ascii=False)

display(pd.DataFrame([status]).T.rename(columns={0: "value"}))
print(f"Saved prompt experiment record: {PROMPT_RESULTS_PATH}")
print(f"Saved Phase 09 status: {STATUS_PATH}")


,value
phase,09_prompt_engineering
required_model,Qwen2.5-7B-Instruct
model_catalog_checked,True
exact_model_available,False
matching_qwen_model_ids,[]
openstax_generative_ai_permission_confirmed,False
retrieved_context_source,data/processed/introduction_to_business_bm25_r...
retrieval_type,BM25 only
retrieved_context_is_real,True
hybrid_or_reranked_context_available,False


Saved prompt experiment record: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_prompt_engineering_results.json
Saved Phase 09 status: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_prompt_engineering_status.json


## Outcome and next controlled execution

This execution demonstrates the eight grounded prompt designs against real BM25-retrieved business context and records their design-level trade-offs. If the exact model or permission preflight fails, it intentionally shows no generated answers rather than using a fallback. After explicit OpenStax permission is obtained and Qwen2.5-7B-Instruct is available, rerun the notebook unchanged to obtain real, auditable outputs. LangGraph is deliberately out of scope for this phase.
